# 06. Поведенческая сегментация и Look-Alike

> **Краткое резюме для руководителя**: Этот ноутбук группирует компании по их поведенческому профилю (частота транзакций, средний чек, направленность потоков, динамика контрагентов). Алгоритм K-Means автоматически находит оптимальное число сегментов. Для каждого сегмента описывается типичный профиль. Дополнительно вычисляется Look-Alike Score — оценка похожести каждой компании на лучших клиентов (топ-10% по обороту), что помогает находить потенциальных ключевых клиентов.

**Процесс**:
1. Загружаем анализированный граф
2. Вычисляем поведенческие признаки (5 метрик)
3. K-Means кластеризация с автоподбором k (по silhouette score)
4. Анализ сегментов: средние значения, распределение
5. Look-Alike scoring: оценка похожести на лучших клиентов
6. Кросс-табуляция ОКВЭД × сегменты

**Предпосылки**: Запустите `03_graph_analysis.ipynb` (или `05_industry_analysis.ipynb` для ОКВЭД-метрик).

---

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.graph_builder import compute_edge_metrics
from src.analysis import (
    compute_behavioral_features,
    cluster_behavioral_segments,
    compute_lookalike_scores,
)

## Загрузка графа

In [ ]:
graph_path = config.OUTPUT_FILTERED_GRAPH_PICKLE
print(f'Loading graph from: {graph_path}')

with open(graph_path, 'rb') as f:
    G = pickle.load(f)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## Поведенческие признаки

Для каждого узла с транзакционными рёбрами вычисляются 5 признаков:

| Признак | Описание | Интерпретация |
|---------|----------|---------------|
| **monthly_tx_count_avg** | Среднее количество транзакций в месяц | Высокое = активный клиент |
| **monthly_amount_avg** | Средний месячный оборот | Высокое = крупный клиент |
| **direction_ratio** | Исходящий / (Входящий + Исходящий) | 0 = только получает, 1 = только отправляет, 0.5 = баланс |
| **counterparty_growth_rate** | Рост числа контрагентов (во 2-й половине периода vs 1-ю) | >0 = расширяется, <0 = сужается |
| **new_counterparty_share** | Доля «новых» контрагентов (впервые появились в поздний период) | Высокое = активно ищет новых партнёров |

In [ ]:
features = compute_behavioral_features(G)

print(f'Behavioral features computed for {len(features)} nodes')
print(f'\nFeature statistics:')
display(features.describe())

## K-Means сегментация

Алгоритм **K-Means** группирует компании по похожести поведенческих профилей:
1. Признаки нормализуются (StandardScaler) чтобы все были в одном масштабе
2. Перебираются варианты k (количество сегментов) от 3 до 10
3. Оптимальное k выбирается по **silhouette score** (мера качества кластеризации: -1 = плохо, +1 = идеально)

**Результат**: каждый узел получает номер сегмента (behavioral_segment).

In [ ]:
segmented = cluster_behavioral_segments(features)

n_segments = segmented['behavioral_segment'].nunique()
print(f'Optimal segments found: {n_segments}')
print(f'\nSegment distribution:')
print(segmented['behavioral_segment'].value_counts().sort_index())

## Профили сегментов

Средние значения признаков для каждого сегмента помогают понять «типичного представителя»:
- Сегмент с высоким monthly_amount_avg = крупные клиенты
- Сегмент с высоким direction_ratio = преимущественно отправители (поставщики)
- Сегмент с высоким new_counterparty_share = быстрорастущие компании

In [ ]:
# Средние значения по сегментам
feature_cols = ['monthly_tx_count_avg', 'monthly_amount_avg', 'direction_ratio',
                'counterparty_growth_rate', 'new_counterparty_share']
segment_profiles = segmented.groupby('behavioral_segment')[feature_cols].mean()

print('Segment profiles (mean values):')
display(segment_profiles)

In [ ]:
# Визуализация: размер сегментов + средний оборот
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Размер сегментов
seg_counts = segmented['behavioral_segment'].value_counts().sort_index()
axes[0].bar(seg_counts.index.astype(str), seg_counts.values, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Number of Nodes')
axes[0].set_title('Segment Size Distribution')

# Средний оборот по сегментам
axes[1].bar(segment_profiles.index.astype(str),
            segment_profiles['monthly_amount_avg'],
            edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Avg Monthly Amount')
axes[1].set_title('Average Monthly Turnover by Segment')

plt.tight_layout()
plt.show()

In [ ]:
# Radar chart (spider plot) для сравнения сегментов
from sklearn.preprocessing import MinMaxScaler

# Нормализуем профили для сравнения
scaler = MinMaxScaler()
norm_profiles = pd.DataFrame(
    scaler.fit_transform(segment_profiles),
    index=segment_profiles.index,
    columns=segment_profiles.columns,
)

labels = ['tx_count', 'amount', 'direction', 'growth', 'new_share']
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = plt.cm.tab10.colors

for i, (seg_id, row) in enumerate(norm_profiles.iterrows()):
    values = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, values, 'o-', label=f'Segment {seg_id}',
            color=colors[i % len(colors)], linewidth=2)
    ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_title('Segment Profiles (normalized)', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.show()

## Look-Alike Scoring

**Цель**: найти компании, похожие на «лучших клиентов» (топ-10% по обороту).

**Алгоритм**:
1. Определяем «лучших клиентов» — топ-10% по суммарному обороту
2. Вычисляем центроид (средний профиль) лучших клиентов
3. Для каждой компании считаем евклидово расстояние до центроида
4. Score = 1 / (1 + distance) — чем ближе к 1, тем больше компания похожа на лучших

**Интерпретация**: lookalike_score > 0.8 = компания ведёт себя как лучший клиент, но пока не в топе по обороту → потенциал для развития.

In [ ]:
scored = compute_lookalike_scores(segmented, G)

if 'lookalike_score' in scored.columns:
    print(f'Look-alike scores computed for {scored["lookalike_score"].notna().sum()} nodes')
    print(f'\nScore distribution:')
    print(scored['lookalike_score'].describe())

    # Топ-20 prospects (не в топ-10% по обороту, но с высоким lookalike_score)
    print(f'\nTop-20 look-alike prospects:')
    top_prospects = scored.nlargest(20, 'lookalike_score')
    display_cols = ['monthly_tx_count_avg', 'monthly_amount_avg',
                    'direction_ratio', 'behavioral_segment', 'lookalike_score']
    display(top_prospects[[c for c in display_cols if c in top_prospects.columns]])
else:
    print('Look-alike scores not computed (insufficient data).')

In [ ]:
# Распределение look-alike scores
if 'lookalike_score' in scored.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(scored['lookalike_score'].dropna(), bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(0.8, color='red', linestyle='--', label='High similarity threshold (0.8)')
    ax.set_xlabel('Look-Alike Score')
    ax.set_ylabel('Number of Nodes')
    ax.set_title('Distribution of Look-Alike Scores')
    ax.legend()
    plt.tight_layout()
    plt.show()

    high_score = (scored['lookalike_score'] >= 0.8).sum()
    print(f'Nodes with lookalike_score >= 0.8: {high_score}')

## Кросс-табуляция: ОКВЭД × сегменты

Как распределены сегменты по отраслям? Помогает ответить на вопросы:
- Есть ли отрасли, где преобладает один поведенческий тип?
- Есть ли неожиданные комбинации (напр. мелкий сегмент в крупной отрасли)?

In [ ]:
# Добавляем ОКВЭД к segmented
okved_series = pd.Series(
    {n: d.get('okved_code', '00') for n, d in G.nodes(data=True)},
    name='okved_code',
)
cross_df = scored.join(okved_series, how='left')
cross_df = cross_df[cross_df['okved_code'] != '00']  # exclude individuals

if not cross_df.empty and 'behavioral_segment' in cross_df.columns:
    cross_tab = pd.crosstab(cross_df['okved_code'], cross_df['behavioral_segment'],
                             margins=True, margins_name='Total')
    print('OKVED x Segment cross-tabulation:')
    display(cross_tab)
else:
    print('Insufficient data for cross-tabulation.')

## Сохранение результатов

In [ ]:
# Сохраняем поведенческие данные
behavioral_path = os.path.join(config.DATA_DIR, 'behavioral_segments.parquet')
scored.to_parquet(behavioral_path)
print(f'Behavioral segments saved: {behavioral_path}')

---

## Глоссарий терминов

| Термин | Описание |
|--------|----------|
| **Behavioral Segment** | Группа компаний с похожим поведенческим профилем (частота, объём, направленность транзакций) |
| **K-Means** | Алгоритм кластеризации: разделяет данные на k групп по минимуму суммы расстояний до центроидов |
| **Silhouette Score** | Метрика качества кластеризации (-1 до +1): чем выше, тем лучше разделение на группы |
| **StandardScaler** | Нормализация: переводит каждый признак к среднему = 0, стд. отклонение = 1 |
| **direction_ratio** | Доля исходящих транзакций в общем потоке: 0 = только получает, 1 = только отправляет |
| **counterparty_growth_rate** | Изменение числа контрагентов со временем: >0 = рост, <0 = сокращение |
| **new_counterparty_share** | Доля новых контрагентов (впервые появились в поздний период) |
| **Look-Alike Score** | Мера похожести на «лучших клиентов»: 1 = идентичный профиль, 0 = максимально далёк |
| **Центроид** | Средний поведенческий профиль группы «лучших клиентов» (эталон для сравнения) |
| **Евклидово расстояние** | Геометрическое расстояние в пространстве признаков: чем меньше, тем больше похожесть |

### Практическое применение

1. **Для relationship managers**: Компании с lookalike_score > 0.8 — потенциальные ключевые клиенты, похожие по поведению на лучших
2. **Для risk-аналитиков**: Сегменты с аномально высоким direction_ratio или new_counterparty_share могут содержать подозрительные компании
3. **Для стратегии**: Кросс-табуляция ОКВЭД × сегменты показывает, какие отрасли недостаточно развиты как клиентская база

---

**Анализ завершён.** Все результаты сохранены в `data/`.